# CLACELL Example Workflow

This Notebook demonstrates the end-to-end workflow for cell-type classification using **CLACELL**. 

The workflow covers:

1. **Marker-based Initial Annotation** using `MarkerAnnotator`
2. **Data Preprocessing** via `preprocess_data`
3. **Donor-wise Train/Test Splitting**
4. **Classification via Custom Ensemble** using `CellClassifier`
5. **Classification via Conditional Autoencoder** using `ConditionalCellClassifier`

In [ ]:
import json
import pickle
import anndata as ad
from sklearn.metrics import f1_score

from clacell import (
    MarkerAnnotator,
    preprocess_data,
    CellClassifier,
    ConditionalCellClassifier,
)

In [ ]:
# Load data
adata = ad.io.read_h5ad('path/to/your/anndata_file.h5ad')

# Optional: load out-of-distribution data
adata_ood = ad.io.read_h5ad('path/to/your/ood_anndata_file.h5ad')

## 1. Marker-Based Cell Annotation (`MarkerAnnotator`)

The `MarkerAnnotator` uses a dictionary of known cell-type marker genes in JSON format to assign initial cell-type labels for the training. The resulting annotations are stored in `adata.obs['scumi-annotation']`.

In [ ]:
# Specify custom marker gene JSON or use the ones provided by scumi
with open("../python_reimplementation_of_scumi/human_pbmc_marker.json", "r", encoding="utf-8") as file:
    marker_genes = json.load(file)

# Initialize annotator with marker genes and generate cell-type annotations
annotator = MarkerAnnotator(marker_genes)
adata = annotator.annotate(adata)

# Out-of-distribution
adata_ood = annotator.annotate(adata_ood)

# The labels are now in adata.obs['scumi-annotation']

## 2. Data Preprocessing

Filter and normalize the gene expression data. You don't have to use CLACELL's preprocessing pipeline, you can use your own. It should include a normalization, log transformation and filtering for highly variable genes, as these steps are crucial for the performance of the CLACELL models.

In [ ]:
adata_preprocessed = preprocess_data(adata)

# Out-of-distribution
adata_ood_preprocessed = preprocess_data(adata_ood)

## 3. Donor-Wise Train / Test Split

To evaluate the model on an in-distribution test set, you have to split the dataset. We highly recommend to split the dataset by the batch column (e.g. donors) to avoid data leakage. Furthermore, you have to convert your anndata objects to pandas Dataframes (X) and Series (y) to use our classifiers.

In [ ]:
# Split the dataset in train and test data to evaluate the model. Example:
donor_train = ['637C', 'A35', 'A36', 'D503'] # Change them to your train donors
donor_test = ['621B', 'D496'] # Change them to your test donors

adata_train = adata_preprocessed[
    adata_preprocessed.obs["Donor"].isin(donor_train)
].copy()

adata_test = adata_preprocessed[
    adata_preprocessed.obs["Donor"].isin(donor_test)
].copy()

In [ ]:
# Convert the train and test split to Dataframes (X) and Series (y)
X_train = adata_train.to_df()
y_train = adata_train.obs['scumi-annotation']

X_test = adata_test.to_df()
y_test = adata_test.obs['scumi-annotation']

# Out-of-distribution
X_ood = adata_ood_preprocessed.to_df()
y_ood = adata_ood_preprocessed.obs['scumi-annotation']

## 4. Classification via Custom Ensemble (`CellClassifier`)

`CellClassifier` fits a custom ensemble designed to maximize prediction stability and minimize the standard deviation of predictions across feature importance dropout steps.

### Initialization Parameters:
* `dropout_steps`: List/Tuple of Feature Importance Dropout thresholds in percentage (default: `(0.0, 0.0075, 0.015, 0.025)`).
* `n_iter_search`: Number of hyperparameter optimization iterations (default: `30`).
* `random_state`: Random seed for reproducibility used in Random Forest and LinearSVC (default: `None`).

In [ ]:
# Initialize classifier with custom or default parameters
classifier = CellClassifier(
    dropout_steps=(0.0, 0.0075, 0.015, 0.025),
    n_iter_search=30,
    random_state=None
)

# This is equal to using minimal parameters:
# classifier = CellClassifier()

### 4.1 Hyperparameter Tuning

Tune parameters using Bayesian Optimization (`bayes_search`) or Randomized Search (`random_search`).

#### Hyperparameters:
* `X_train`: Train Dataframe (required).
* `y_train`: Train Series (required).
* `X_test`: In-distribution test Dataframe (default: `None`).
* `y_test`: In-distribution test Series (default: `None`).
* `n_jobs`: Specifies the number of parallel jobs (default: `1`).

> **Important Note on Data Leakage:**  
> If you specify `X_test` and `y_test` during `bayes_search` or `random_search`, the method combines train and test sets after the search to retrain the final model on the full dataset. Do **not** re-use `X_test` / `y_test` for final evaluation if passed here. To preserve an unseen test set, pass only `X_train` and `y_train` into the search method.

In [ ]:
# Option A: Full Bayesian Search with evaluation set during tuning
classifier.bayes_search(X_train, y_train, X_test, y_test, n_jobs=3)

# Option B: Bayesian Search using only training data (Recommended if reusing X_test for evaluation)
# classifier.bayes_search(X_train, y_train)

# Option C: Randomized Search alternative
# classifier.random_search(X_train, y_train, X_test, y_test, n_jobs=3)

### 4.2 Direct Model Training (Pre-defined Parameters)

If optimal hyperparameters for the LinearSVC are already known, train the classifier directly without running search algorithms.

In [ ]:
# Train directly with custom hyperparameters (e.g., regularization strength C and tolerance tol). 
# You can add as many hyperparameters for the LinearSVC as you like.
classifier.train(X_train, y_train, C=0.001, tol=0.001)

### 4.3 Model Evaluation

Evaluate classification performance on an in-distribution and optional an out-of-distribution test set

#### Hyperparameters:
* `X_test`: In-distribution test Dataframe (required).
* `y_test`: In-distribution test Series (required).
* `labels`: Specifies where the labels are if an Anndata object is provided for X_ood (default: `scumi-annotation`).
* `X_ood`: Out-of-distribution test Dataframe (default: `None`).
* `y_ood`: Out-of-distribution test Series (default: `None`).
* `feature_importances`: Feature Importances used for the Feature Importance Dropout test. If None, then the test will be skipped (default: `None`).
* `log_to_console`: Flag if the results should be logged to the console (default: `True`).
* `log_to_file`: Flag if the results should be logged to a log file (default: `True`).

#### Output

The method returns a Dataframe with a Multi index to better access every metric

In [ ]:
# Specify your own Feature Importance for Feature Importance Dropout test or use our provided Feature Importance.
# It is based on marker genes and differential gene expressions
with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

# Evaluate model metrics and log findings
results = classifier.evaluate(
    X_test,
    y_test,
    X_ood=X_ood,
    y_ood=y_ood,
    feature_importances=feature_importance,
    log_to_console=True,
    log_to_file=True
)

print(results)

### 4.4 Prediction

Predict cell types for new data

In [ ]:
predictions = classifier.predict(X_test)
print(f"Macro F1: {f1_score(y_test, predictions, average="macro")}")

## 5. Classification via Conditional Autoencoder (`ConditionalCellClassifier`)

`ConditionalCellClassifier` uses a Conditional Autoencoder architecture designed to maximize generalization performance on out-of-distribution data.

### Initialization Parameters:
* `n_iter_search`: Number of hyperparameter optimization iterations (default: `30`).
* `random_state`: Random seed for reproducibility used in Random Forest and LinearSVC (default: `None`).

In [ ]:
# Initialize classifier with custom or default parameters
cond_classifier = ConditionalCellClassifier(
    n_iter_search=30,
    random_state=None
)

# This is equal to using minimal parameters:
# classifier = ConditionalCellClassifier()

### 5.1 Hyperparameter Tuning

Tune parameters using Bayesian Optimization (`bayes_search`) or Randomized Search (`random_search`).

#### Hyperparameters:
* `X_train`: Train Dataframe (required).
* `y_train`: Train Series (required).
* `donor_train`: The donor ids for the train split (required).
* `X_test`: In-distribution test Dataframe (default: `None`).
* `y_test`: In-distribution test Series (default: `None`).
* `donor_test`: The donor ids for the test split (default: `None`).
* `n_jobs`: Specifies the number of parallel jobs (default: `1`).

> **Important Note on Data Leakage:**  
> If you specify `X_test` and `y_test` during `bayes_search` or `random_search`, the method combines train and test sets after the search to retrain the final model on the full dataset. Do **not** re-use `X_test` / `y_test` for final evaluation if passed here. To preserve an unseen test set, pass only `X_train` and `y_train` into the search method.

In [ ]:
# Get donors
train_donor = adata_train.obs['Donor']
test_donor = adata_test.obs['Donor']

# Option A: Full Bayesian Search with evaluation set during tuning
cond_classifier.bayes_search(X_train, y_train, train_donor, X_test, y_test, test_donor, n_jobs=3)

# Option B: Bayesian Search using only training data (Recommended if reusing X_test for evaluation)
# cond_classifier.bayes_search(X_train, y_train)

# Option C: Randomized Search alternative
# cond_classifier.random_search(X_train, y_train, X_test, y_test, n_jobs=3)

### 5.2 Direct Model Training (Pre-defined Parameters)

If optimal hyperparameters for the LinearSVC are already known, train the classifier directly without running search algorithms.

In [ ]:
# Get donors
train_donor = adata_train.obs['Donor']

# Train directly with custom hyperparameters (e.g., regularization strength C and tolerance tol). 
# You can add as many hyperparameters for the LinearSVC as you like.
cond_classifier.train(X_train, y_train, train_donor, C=0.001, tol=0.001)

### 5.3 Model Evaluation

Evaluate classification performance on an in-distribution and optional an out-of-distribution test set

#### Hyperparameters:
* `X_test`: In-distribution test Dataframe (required).
* `y_test`: In-distribution test Series (required).
* `labels`: Specifies where the labels are if an Anndata object is provided for X_ood (default: `scumi-annotation`).
* `X_ood`: Out-of-distribution test Dataframe or Anndata object (default: `None`).
* `y_ood`: Out-of-distribution test Series (default: `None`).
* `feature_importances`: Feature Importances used for the Feature Importance Dropout test. If None, then the test will be skipped (default: `None`).
* `log_to_console`: Flag if the results should be logged to the console (default: `True`).
* `log_to_file`: Flag if the results should be logged to a log file (default: `True`).

#### Output

The method returns a Dataframe with a Multi index to better access every metric

In [ ]:
# Specify your own Feature Importance for Feature Importance Dropout test or use our provided Feature Importance.
# It is based on marker genes and differential gene expressions
with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

# Evaluate model metrics and log findings
results = cond_classifier.evaluate(
    X_test,
    y_test,
    X_ood=X_ood,
    y_ood=y_ood,
    feature_importances=feature_importance,
    log_to_console=True,
    log_to_file=True
)

print(results)

### 5.4 Prediction

Predict cell types for new data

In [ ]:
predictions = cond_classifier.predict(X_test)
print(f"Macro F1: {f1_score(y_test, predictions, average="macro")}")